# Prepare Head Metrics Dataset

This notebook converts a user-provided head-metrics table to the 47-feature schema used in the FAST experiments. Edit the paths and mapping config below, then run all cells.

## 1. Configure Paths

Set the raw input files, standardized output files, and the JSON mapping config. The mapping file can use either `target_to_source_mapping` or `source_to_target_mapping`.

In [ ]:
from pathlib import Path
import json
import sys

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'notebook':
    NOTEBOOK_DIR = Path('notebook')

sys.path.insert(0, str((NOTEBOOK_DIR / 'scripts').resolve()))

TRAIN_INPUT = Path('path/to/your_train_dataset.xlsx')
TEST_INPUT = Path('path/to/your_test_dataset.xlsx')
TRAIN_OUTPUT = NOTEBOOK_DIR / 'outputs' / 'train_standardized.csv'
TEST_OUTPUT = NOTEBOOK_DIR / 'outputs' / 'test_standardized.csv'
TRANSFORM_CONFIG = NOTEBOOK_DIR / 'configs' / 'transform_config_template.json'

TRAIN_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print(TRAIN_OUTPUT)
print(TEST_OUTPUT)

## 2. Review Required Features

The pipeline expects the same 47 features used by the FAST experiments: 46 head-motion variables plus `vr_system_ordinal`.

In [ ]:
from head_metrics_schema import FEATURE_COLUMNS, FEATURE_FAMILIES

print(f'Required features: {len(FEATURE_COLUMNS)}')
for i, feature in enumerate(FEATURE_COLUMNS, start=1):
    print(f'{i:02d}. {feature}')

print('\nFeature families:')
for family, features in FEATURE_FAMILIES.items():
    print(f'- {family}: {len(features)} features')

## 3. Inspect And Edit The Mapping Config

If your columns already use the expected names, the template works as-is. Otherwise, edit `target_to_source_mapping` so each required FAST feature points to the corresponding column in your dataset.

In [ ]:
config = json.loads(TRANSFORM_CONFIG.read_text(encoding='utf-8'))
print(json.dumps({k: config[k] for k in ['target_column', 'passthrough_columns', 'fill_missing_features']}, indent=2))
print('\nFirst 5 mappings:')
for idx, item in enumerate(config['target_to_source_mapping'].items()):
    if idx >= 5:
        break
    print(item)

## 4. Transform Train And Test Datasets

This cell reads the raw files, applies the mapping, validates that all required features are present and numeric, and writes standardized CSV files.

In [ ]:
from transform_head_metrics import (
    build_target_to_source_mapping,
    read_table,
    standardize_head_metrics,
    validate_standardized_dataset,
    write_table,
)

def transform_one(input_path, output_path):
    raw = read_table(input_path)
    standardized = standardize_head_metrics(
        raw,
        target_to_source_mapping=build_target_to_source_mapping(config),
        passthrough_columns=config.get('passthrough_columns', []),
        target_column=config.get('target_column'),
        fill_missing_features=bool(config.get('fill_missing_features', False)),
        missing_value=float(config.get('missing_value', 0.0)),
    )
    validation = validate_standardized_dataset(standardized, config.get('target_column'))
    if validation['missing_features'] or validation['non_numeric_features']:
        raise ValueError(validation)
    write_table(standardized, output_path)
    return validation

train_validation = transform_one(TRAIN_INPUT, TRAIN_OUTPUT)
test_validation = transform_one(TEST_INPUT, TEST_OUTPUT)

print('Train validation:')
print(json.dumps(train_validation, indent=2))
print('\nTest validation:')
print(json.dumps(test_validation, indent=2))

## 5. Preview Standardized Output

Check that the standardized files contain the expected feature columns and target.

In [ ]:
import pandas as pd

preview = pd.read_csv(TRAIN_OUTPUT)
display(preview.head())
display(preview[config['target_column']].value_counts().sort_index())